# nofe token visualisation
PCA / k-means / SLIC views of the tubelet tokens (and last hidden state) for each run.
Needs `src/utils/viz.py` and trained runs in `runs/`.

In [ ]:
import os, sys, random
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)

import torch
from src.dataset.dataset_utils import get_dataset
from src.dataset.clips import build_manifest
from src.utils import viz

## Settings

In [ ]:
RUNS = {                      # name -> run folder (delete the ones you have not trained)
    "baseline": "runs/baseline",
    "hybrid":   "runs/hybrid",
    "full":     "runs/full",
}
N_CLIPS = 3                   # random clips to show
K = 6                         # k-means clusters
SEED = 0
# Or set explicit clips: CLIPS = ["path/a.mp4", "path/b.mp4"]
CLIPS = None

## Pick clips

In [ ]:
if CLIPS is None:
    items = build_manifest(get_dataset("kinetics-train-5per"))
    CLIPS = [str(p) for p, _ in random.Random(SEED).sample(items, N_CLIPS)]
clips = {Path(p).name: viz.load_clip(p) for p in CLIPS}
list(clips)

## Load models

In [ ]:
models = {name: viz.load_model(name, folder) for name, folder in RUNS.items()}
for name, m in models.items():
    print(name, "alpha =", viz.get_alpha(m))

## Embedding tokens (output of the tubelet embedding)

In [ ]:
for cname, x in clips.items():
    for mname, m in models.items():
        tok = viz.get_tokens(m, x)
        viz.visualize(tok["embed"], x[0], k=K, title=f"{mname} embedding - {cname}")
        if "nofe" in tok:
            viz.visualize(tok["nofe"], x[0], k=K, title=f"{mname} alpha*nofe only - {cname}")

## Last hidden state (more semantic)

In [ ]:
for cname, x in clips.items():
    for mname, m in models.items():
        tok = viz.get_tokens(m, x, last_hidden=True)
        viz.visualize(tok["last"], x[0], k=K, title=f"{mname} last hidden - {cname}")

## How big is the nofe contribution? (hybrid)

In [ ]:
if "hybrid" in models:
    for cname, x in clips.items():
        tok = viz.get_tokens(models["hybrid"], x)
        print(cname, "norm(alpha*nofe) / norm(orig) =", round(viz.norm_ratio(tok["nofe"], tok["orig"]), 4))